# 🏥 Insurance Customer Churn Prediction

**Author:** Yuxuan Li  
**Tools:** Python, pandas, scikit-learn, matplotlib, seaborn  
**Dataset:** Simulated insurance customer data (3,000 customers)

---

## 📌 Project Overview

Customer churn occurs when a policyholder cancels or fails to renew their insurance. This project builds an end-to-end machine learning pipeline to:
- Explore and understand the data
- Engineer meaningful features
- Apply **PCA** for dimensionality reduction
- Train and compare **3 machine learning models**
- Segment customers by churn risk for business action

---

## 📋 Table of Contents
1. [Import Libraries](#1-import-libraries)
2. [Load & Explore Data](#2-load--explore-data)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis)
4. [Feature Engineering](#4-feature-engineering)
5. [Train/Test Split & Scaling](#5-traintest-split--scaling)
6. [PCA – Dimensionality Reduction](#6-pca--dimensionality-reduction)
7. [Model Training & Evaluation](#7-model-training--evaluation)
8. [Model Comparison](#8-model-comparison)
9. [Feature Importance](#9-feature-importance)
10. [Business Risk Segmentation](#10-business-risk-segmentation)
11. [Conclusions & Recommendations](#11-conclusions--recommendations)


---
## 1. Import Libraries

We start by importing all necessary Python libraries:
- **pandas / numpy**: data manipulation and numerical operations
- **matplotlib / seaborn**: data visualization
- **scikit-learn**: machine learning models and utilities


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

print('✅ All libraries imported successfully!')

> **Note:** If any library is missing, install it with:
> ```bash
> pip install pandas numpy matplotlib seaborn scikit-learn
> ```


---
## 2. Load & Explore Data

We load the insurance churn dataset and take a first look at its structure, size, and key statistics.


In [ ]:
# Load the dataset
df = pd.read_csv('insurance_churn.csv')

print(f"Dataset shape: {df.shape}")
print(f"Number of customers: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")
print(f"\nChurn rate: {df['churn'].mean():.1%}")
print(f"Retained: {(df['churn'] == 0).sum()} customers")
print(f"Churned:  {(df['churn'] == 1).sum()} customers")

> **Observation:** The dataset has a churn rate of ~20.7%, which is realistic for the insurance industry. This is a moderately imbalanced dataset, so we will use **ROC-AUC** as our primary evaluation metric alongside accuracy.


In [ ]:
# Preview the first 5 rows
df.head()

In [ ]:
# Check data types and missing values
print("Data types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())

> **Observation:** No missing values found — the dataset is clean and ready for analysis. Features include a mix of numeric (age, premium, claims) and categorical (policy type, region) variables.


In [ ]:
# Summary statistics for numeric columns
df.describe().round(2)

> **Key observations from summary statistics:**
> - Average customer age: ~46 years
> - Average tenure: ~10 years
> - Average annual premium: ~€2,700
> - Average number of claims: ~0.8 per customer


---
## 3. Exploratory Data Analysis

Before modelling, we visualise the data to understand patterns and relationships between features and churn.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Insurance Churn – Exploratory Data Analysis', fontsize=16, fontweight='bold')

palette = {0: '#2E6DA4', 1: '#E05C2A'}
labels  = {0: 'Retained', 1: 'Churned'}

# 1. Churn distribution
ax = axes[0, 0]
counts = df['churn'].value_counts()
bars = ax.bar(['Retained', 'Churned'], counts.values, color=['#2E6DA4', '#E05C2A'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, str(val), ha='center', fontweight='bold')
ax.set_title('Churn Distribution', fontweight='bold')
ax.set_ylabel('Count')

# 2. Tenure vs Churn
ax = axes[0, 1]
for churn_val, group in df.groupby('churn'):
    ax.hist(group['tenure_years'], bins=15, alpha=0.6, label=labels[churn_val], color=palette[churn_val])
ax.set_title('Tenure Years by Churn', fontweight='bold')
ax.set_xlabel('Tenure (years)')
ax.legend()

# 3. Complaint count vs Churn rate
ax = axes[0, 2]
churn_by_complaint = df.groupby('complaint_count')['churn'].mean() * 100
ax.bar(churn_by_complaint.index, churn_by_complaint.values, color='#E05C2A', edgecolor='white')
ax.set_title('Churn Rate by Complaint Count', fontweight='bold')
ax.set_xlabel('Number of Complaints')
ax.set_ylabel('Churn Rate (%)')

# 4. Policy type vs Churn rate
ax = axes[1, 0]
churn_by_policy = df.groupby('policy_type')['churn'].mean() * 100
ax.bar(churn_by_policy.index, churn_by_policy.values,
       color=['#2E6DA4', '#4A9DBF', '#E05C2A', '#F0A060'], edgecolor='white')
ax.set_title('Churn Rate by Policy Type', fontweight='bold')
ax.set_xlabel('Policy Type')
ax.set_ylabel('Churn Rate (%)')

# 5. Satisfaction score vs Churn rate
ax = axes[1, 1]
churn_by_sat = df.groupby('satisfaction_score')['churn'].mean() * 100
ax.bar(churn_by_sat.index.astype(str), churn_by_sat.values, color='#E05C2A', edgecolor='white')
ax.set_title('Churn Rate by Satisfaction Score', fontweight='bold')
ax.set_xlabel('Satisfaction Score (1=Low, 5=High)')
ax.set_ylabel('Churn Rate (%)')

# 6. Premium increase vs Churn
ax = axes[1, 2]
for churn_val, group in df.groupby('churn'):
    ax.hist(group['premium_increase_pct'], bins=15, alpha=0.6, label=labels[churn_val], color=palette[churn_val])
ax.set_title('Premium Increase % by Churn', fontweight='bold')
ax.set_xlabel('Premium Increase (%)')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/01_eda.png', dpi=150, bbox_inches='tight')
plt.show()

> **Key EDA findings:**
> - **Complaints are a strong churn signal** — customers with 2–3 complaints churn at much higher rates
> - **Shorter tenure = higher churn** — new customers are most at risk
> - **Low satisfaction scores** correlate strongly with churn
> - **Higher premium increases** push customers to leave
> - Policy type shows relatively small differences in churn rate


---
## 4. Feature Engineering

We create new features that capture business-relevant patterns not directly visible in the raw data. Good feature engineering often improves model performance significantly.


In [ ]:
df_model = df.copy()

# Step 1: Encode categorical variables as numbers
# Machine learning models require numeric input
le = LabelEncoder()
df_model['policy_type_enc'] = le.fit_transform(df_model['policy_type'])
df_model['region_enc']      = le.fit_transform(df_model['region'])

# Step 2: Create new meaningful features
df_model['claim_frequency']    = df_model['num_claims'] / (df_model['tenure_years'] + 1)
df_model['premium_per_policy'] = df_model['annual_premium'] / df_model['num_policies']
df_model['high_risk']          = ((df_model['complaint_count'] > 0) & (df_model['num_claims'] > 1)).astype(int)

print("✅ New features created:")
print(f"  claim_frequency    : average claims per year (mean = {df_model['claim_frequency'].mean():.2f})")
print(f"  premium_per_policy : premium per policy (mean = €{df_model['premium_per_policy'].mean():.0f})")
print(f"  high_risk          : customers with complaints AND multiple claims = {df_model['high_risk'].sum()} ({df_model['high_risk'].mean():.1%})")

> **Why these features?**
> - `claim_frequency` captures how often a customer files claims relative to how long they've been with us — a better signal than raw claim count
> - `premium_per_policy` tells us the cost burden per product, which may drive switching behaviour
> - `high_risk` is a composite flag combining two strong churn signals: complaints and frequent claims


---
## 5. Train/Test Split & Scaling

We split the data into training and test sets, then standardise the features. Standardisation is important because PCA and Logistic Regression are sensitive to feature scale.


In [ ]:
feature_cols = [
    'age', 'tenure_years', 'num_policies', 'annual_premium',
    'num_claims', 'claim_amount', 'complaint_count', 'contacted_support',
    'premium_increase_pct', 'satisfaction_score', 'years_since_last_claim',
    'digital_engagement', 'policy_type_enc', 'region_enc',
    'claim_frequency', 'premium_per_policy', 'high_risk'
]

X = df_model[feature_cols]
y = df_model['churn']

# Split: 80% training, 20% testing
# stratify=y ensures the churn ratio is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardise: transform each feature to have mean=0, std=1
# We fit ONLY on training data to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # use same scaler on test data

print(f"✅ Data split complete:")
print(f"  Training set : {X_train.shape[0]} customers ({y_train.mean():.1%} churn rate)")
print(f"  Test set     : {X_test.shape[0]} customers ({y_test.mean():.1%} churn rate)")
print(f"  Features     : {len(feature_cols)}")

> **Why stratify?** Without stratification, the random split might put most churners in one set, causing the model to be trained or evaluated on unrepresentative data. Stratification ensures both sets have the same churn rate (~20.7%).

> **Why fit scaler only on training data?** If we scaled using the full dataset before splitting, information from the test set would leak into training. This is called **data leakage** and would give artificially inflated results.


---
## 6. PCA – Dimensionality Reduction

**Principal Component Analysis (PCA)** is a technique that reduces the number of features while retaining as much information as possible. It transforms correlated features into a smaller set of uncorrelated components.

**Why use PCA?**
- Removes multicollinearity between features
- Reduces noise and overfitting
- Speeds up model training
- Improves Logistic Regression performance


In [ ]:
# First, fit PCA on ALL components to see the explained variance curve
pca_full = PCA()
pca_full.fit(X_train_scaled)

# Calculate cumulative explained variance
explained_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Find how many components explain 95% of variance
n_components_95 = np.argmax(explained_variance >= 0.95) + 1

print(f"✅ PCA Analysis:")
print(f"  Total features           : {X_train_scaled.shape[1]}")
print(f"  Components for 95% variance : {n_components_95}")
print(f"  Variance explained per component:")
for i, var in enumerate(pca_full.explained_variance_ratio_[:5], 1):
    print(f"    PC{i}: {var:.1%}")
print(f"    ...")

In [ ]:
# Apply PCA keeping components that explain 95% variance
pca = PCA(n_components=n_components_95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

# Also apply 2D PCA for visualization purposes
pca_2d = PCA(n_components=2, random_state=42)
X_train_2d = pca_2d.fit_transform(X_train_scaled)

# Plot PCA results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('PCA – Dimensionality Reduction', fontsize=14, fontweight='bold')

# Scree plot: cumulative explained variance
ax = axes[0]
components = np.arange(1, len(explained_variance) + 1)
ax.plot(components, explained_variance * 100, 'o-', color='#2E6DA4', lw=2, markersize=5)
ax.axhline(y=95, color='#E05C2A', linestyle='--', lw=1.5, label='95% threshold')
ax.axvline(x=n_components_95, color='#2EAA6D', linestyle='--', lw=1.5,
           label=f'{n_components_95} components selected')
ax.fill_between(components, explained_variance * 100, alpha=0.1, color='#2E6DA4')
ax.set_title('Cumulative Explained Variance', fontweight='bold')
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.legend()
ax.grid(alpha=0.3)

# 2D scatter: visualise customer segments
ax = axes[1]
colors_pca = {0: '#2E6DA4', 1: '#E05C2A'}
labels_pca  = {0: 'Retained', 1: 'Churned'}
for churn_val in [0, 1]:
    mask = y_train == churn_val
    ax.scatter(X_train_2d[mask, 0], X_train_2d[mask, 1],
               c=colors_pca[churn_val], label=labels_pca[churn_val],
               alpha=0.4, s=20, edgecolors='none')
var1 = pca_2d.explained_variance_ratio_[0] * 100
var2 = pca_2d.explained_variance_ratio_[1] * 100
ax.set_title('PCA – 2D Visualization of Customer Segments', fontweight='bold')
ax.set_xlabel(f'PC1 ({var1:.1f}% variance)')
ax.set_ylabel(f'PC2 ({var2:.1f}% variance)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_pca.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ PCA applied: {X_train_scaled.shape[1]} features → {n_components_95} components")
print(f"   Variance retained: ≥95%")

> **Reading the scree plot:** The elbow of the curve shows where adding more components gives diminishing returns. We select the point where 95% of variance is explained — this balances information retention with dimensionality reduction.

> **Reading the 2D scatter plot:** Even with just 2 components (a simplified view), we can see a degree of separation between churned (orange) and retained (blue) customers — confirming that the features contain genuine predictive signal.


---
## 7. Model Training & Evaluation

We train three different models and evaluate them using:
- **Accuracy**: percentage of correct predictions
- **ROC-AUC**: ability to distinguish churners from non-churners (main metric)
- **CV-AUC**: cross-validation AUC — tests stability across 5 different data splits


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    # Logistic Regression uses PCA-reduced features
    # Tree models work better with original features
    X_tr = X_train_pca if name == 'Logistic Regression' else X_train
    X_te = X_test_pca  if name == 'Logistic Regression' else X_test

    # Train the model
    model.fit(X_tr, y_train)

    # Make predictions
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]  # probability of churning

    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cv  = cross_val_score(
              model,
              X_train_pca if name == 'Logistic Regression' else X_train,
              y_train, cv=5, scoring='roc_auc'
          ).mean()

    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
        'accuracy': acc, 'auc': auc, 'cv_auc': cv
    }

    print(f"  {name:25s}  Accuracy: {acc:.1%}  ROC-AUC: {auc:.3f}  CV-AUC: {cv:.3f}")

> **Why three models?**
> - **Logistic Regression**: simple, interpretable baseline — good with PCA-reduced features
> - **Random Forest**: ensemble of decision trees — robust to outliers, handles non-linear patterns
> - **Gradient Boosting**: sequentially improves on errors — often best overall performance

> **Why ROC-AUC as primary metric?** With ~20% churn rate, a model that predicts "retained" for everyone would get 79% accuracy but be completely useless. ROC-AUC measures actual discriminative power: 0.5 = random, 1.0 = perfect.


---
## 8. Model Comparison

We compare all three models visually using ROC curves, AUC bar charts, and a confusion matrix for the best model.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')

colors_roc = ['#2E6DA4', '#E05C2A', '#2EAA6D']

# ROC Curves
ax = axes[0]
for (name, res), col in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=col, lw=2)
ax.plot([0,1],[0,1],'k--', lw=1, label='Random (AUC=0.500)')
ax.set_title('ROC Curves', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# AUC Bar Chart
names   = list(results.keys())
aucs    = [results[n]['auc']    for n in names]
cv_aucs = [results[n]['cv_auc'] for n in names]
x = np.arange(len(names))
w = 0.35
ax = axes[1]
bars1 = ax.bar(x - w/2, aucs,    w, label='Test AUC', color='#2E6DA4', edgecolor='white')
bars2 = ax.bar(x + w/2, cv_aucs, w, label='CV AUC',   color='#E05C2A', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '
') for n in names], fontsize=9)
ax.set_ylim(0.5, 1.0)
ax.set_title('Test AUC vs Cross-Validation AUC', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=8)

# Confusion Matrix for best model
best_name = max(results, key=lambda n: results[n]['accuracy'])
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned']).plot(
    ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title(f'Confusion Matrix\n{best_name} (Best Model)', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/03_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

> **Reading the confusion matrix:**
> - **Top-left**: correctly predicted retained (True Negatives)
> - **Bottom-right**: correctly predicted churned (True Positives) ✅
> - **Top-right**: predicted churned but actually retained (False Positives)
> - **Bottom-left**: predicted retained but actually churned (False Negatives) ⚠️ — these are missed churners

> In a business context, **False Negatives are costly** — these are churners we missed and couldn't intervene for. A lower false negative rate means better retention.


---
## 9. Feature Importance

Random Forest provides a built-in feature importance score — it measures how much each feature contributes to reducing prediction error across all decision trees.


In [ ]:
rf_model    = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors_fi = ['#E05C2A' if imp > importances.quantile(0.75) else '#2E6DA4'
             for imp in importances.values]
bars = ax.barh(importances.index, importances.values, color=colors_fi, edgecolor='white')
ax.set_title('Feature Importance – Random Forest\n(Orange = Top 25% most important features)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 most important features:")
for i, (feat, val) in enumerate(importances.tail(5).iloc[::-1].items(), 1):
    print(f"  {i}. {feat}: {val:.4f}")

> **Key insight:** The top churn drivers are:
> 1. **complaint_count** — the single strongest predictor of churn
> 2. **premium_increase_pct** — price sensitivity drives switching
> 3. **tenure_years** — loyalty develops over time; new customers are most vulnerable
> 4. **satisfaction_score** — early warning signal before churn happens
> 5. **claim_frequency** — frequent claimers who feel underserved are more likely to leave

> These insights directly inform where to focus retention efforts.


---
## 10. Business Risk Segmentation

Using the Gradient Boosting model, we assign each customer a churn probability score and segment them into four risk tiers for targeted action.


In [ ]:
# Predict churn probability for all customers
df_model['churn_prob'] = results['Gradient Boosting']['model'].predict_proba(
    df_model[feature_cols])[:, 1]

# Segment customers into risk tiers
df_model['risk_segment'] = pd.cut(
    df_model['churn_prob'],
    bins=[0, 0.1, 0.25, 0.5, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical']
)

# Summary table
summary = df_model.groupby('risk_segment', observed=True).agg(
    customers=('churn_prob', 'count'),
    avg_churn_prob=('churn_prob', 'mean'),
    avg_premium=('annual_premium', 'mean'),
    total_premium=('annual_premium', 'sum')
).round(2)

summary['total_premium'] = (summary['total_premium'] / 1000).round(1)
summary.columns = ['Customers', 'Avg Churn Prob', 'Avg Premium (€)', 'Total Premium at Risk (€K)']
print("Customer Risk Segmentation Summary:")
print(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Business Insights: Customer Risk Segmentation', fontsize=14, fontweight='bold')

seg_colors = ['#2EAA6D', '#F0C040', '#E05C2A', '#8B0000']
seg_counts  = df_model['risk_segment'].value_counts()

# Pie chart: customer distribution
axes[0].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
            colors=seg_colors, startangle=90, pctdistance=0.75)
axes[0].set_title('Customer Distribution by Risk Segment', fontweight='bold')

# Bar chart: premium at risk
seg_order = ['Low Risk', 'Medium Risk', 'High Risk', 'Critical']
premium_at_risk = df_model.groupby('risk_segment', observed=True)['annual_premium'].sum() / 1_000
vals = [premium_at_risk.get(s, 0) for s in seg_order]
bars = axes[1].bar(seg_order, vals, color=seg_colors, edgecolor='white')
for bar, val in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'€{val:.0f}K', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Total Annual Premium at Risk by Segment', fontweight='bold')
axes[1].set_ylabel('Total Premium (thousands €)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/05_business_insights.png', dpi=150, bbox_inches='tight')
plt.show()

> **Business interpretation:**
> - Most customers fall in the **Low Risk** segment — standard service applies
> - **High Risk and Critical** customers represent a concentrated amount of premium at risk
> - Prioritising outreach to these segments delivers the highest retention ROI


---
## 11. Conclusions & Recommendations

### ✅ Model Performance Summary

| Model | Accuracy | ROC-AUC | CV-AUC |
|---|---|---|---|
| **Logistic Regression (with PCA)** | **~95%** | **0.994** | **0.992** |
| Gradient Boosting | 96.0% | 0.991 | 0.985 |
| Random Forest | 93.5% | 0.981 | 0.973 |

**Best model: Logistic Regression with PCA** — achieves ~95% accuracy and excellent AUC scores confirmed by cross-validation.

---

### 🔍 Key Churn Drivers

1. **Complaint count** — the single strongest predictor; resolve complaints immediately
2. **Premium increase %** — customers are price-sensitive at renewal; be transparent
3. **Tenure** — new customers (< 3 years) are most vulnerable; invest in onboarding
4. **Satisfaction score** — monitor regularly and act on low scores early
5. **Claim frequency** — frequent claimers who feel underserved will leave

---

### 📋 Recommended Business Actions

| Segment | Action | Priority |
|---|---|---|
| 🔴 Critical (>50% churn prob) | Immediate personal call, loyalty offer | Highest |
| 🟠 High Risk (25–50%) | Proactive outreach before renewal | High |
| 🟡 Medium Risk (10–25%) | Automated satisfaction check | Medium |
| 🟢 Low Risk (<10%) | Standard service | Low |

---

### 💡 Next Steps
- Deploy model as a real-time scoring API for the sales team
- Re-train monthly as new customer data arrives
- A/B test retention interventions to measure actual impact
- Explore additional features: payment history, website engagement, agent interactions
